In [ ]:
# Load Gold Dataset
ml_dataset = spark.read.format("delta").load(
    "/Volumes/workspace/default/kthdv&dtdm/gold/ml_dataset"
)

display(ml_dataset.limit(5))

In [ ]:
# Remove Remaining NULL and duplicate orders
ml_dataset = ml_dataset.dropna().dropDuplicates(["order_id"])

print("Rows:", ml_dataset.count())
ml_dataset.groupBy("label").count().orderBy("label").show()

In [ ]:
# Import Machine Learning Libraries
from pyspark.ml.feature import (
    StringIndexer,
    VectorAssembler,
    StandardScaler
)

from pyspark.ml.classification import (
    LogisticRegression,
    RandomForestClassifier,
    GBTClassifier
)

from pyspark.ml import Pipeline
from pyspark.sql.functions import col, lit, sum, when

from pyspark.ml.evaluation import (
    BinaryClassificationEvaluator,
    MulticlassClassificationEvaluator
)

In [ ]:
# Encode Categorical Features
# payment_type: ML model không hiểu string nên phải Encode
payment_indexer = StringIndexer(
    inputCol="payment_type",
    outputCol="payment_type_index",
    handleInvalid="keep"
)

In [ ]:
# Feature Columns
feature_columns = [
    "payment_type_index",
    "payment_installments",
    "number_of_items",
    "avg_item_price",
    "delivery_time",
    "delivery_delay",
    "shipping_duration",
    "order_total_value",
    "customer_total_orders",
    "customer_total_spent",
    "avg_review_score_customer"
]

In [ ]:
# VectorAssembler   
assembler = VectorAssembler(
    inputCols=feature_columns,
    outputCol="features"
)

In [ ]:
# Scaling
scaler = StandardScaler(
    inputCol="features",
    outputCol="scaled_features",
    withStd=True,
    withMean=False
)

In [ ]:
# Chronological Train/Test Split
# Train on older orders and evaluate on the newest 20% of orders.
dataset_with_epoch = ml_dataset.withColumn(
    "_purchase_epoch",
    col("order_purchase_timestamp").cast("long")
)

split_epoch = dataset_with_epoch.approxQuantile(
    "_purchase_epoch",
    [0.8],
    0.001
)[0]

train_df = dataset_with_epoch.filter(
    col("_purchase_epoch") < split_epoch
)

test_df = dataset_with_epoch.filter(
    col("_purchase_epoch") >= split_epoch
)

print("Split epoch:", split_epoch)
print("Train:", train_df.count())
print("Test :", test_df.count())
train_df.groupBy("label").count().orderBy("label").show()
test_df.groupBy("label").count().orderBy("label").show()

In [ ]:
# Logistic Regression
# Create Model
lr = LogisticRegression(
    featuresCol="scaled_features",
    labelCol="label"
)
# Pipeline 
lr_pipeline = Pipeline(stages=[
    payment_indexer,
    assembler,
    scaler,
    lr
])
# Train 
lr_model = lr_pipeline.fit(train_df)
# Predict
lr_predictions = lr_model.transform(test_df)
display(
    lr_predictions.select(
        "label",
        "prediction",
        "probability"
    )
)

In [ ]:
# Evaluation helpers
roc_auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderROC"
)

pr_auc_evaluator = BinaryClassificationEvaluator(
    labelCol="label",
    rawPredictionCol="rawPrediction",
    metricName="areaUnderPR"
)

def evaluate_predictions(model_name, predictions):
    counts = predictions.agg(
        sum(when((col("label") == 1) & (col("prediction") == 1), 1).otherwise(0)).alias("tp"),
        sum(when((col("label") == 0) & (col("prediction") == 0), 1).otherwise(0)).alias("tn"),
        sum(when((col("label") == 0) & (col("prediction") == 1), 1).otherwise(0)).alias("fp"),
        sum(when((col("label") == 1) & (col("prediction") == 0), 1).otherwise(0)).alias("fn")
    ).first().asDict()

    tp = counts["tp"]
    tn = counts["tn"]
    fp = counts["fp"]
    fn = counts["fn"]
    total = tp + tn + fp + fn

    precision = tp / (tp + fp) if tp + fp else 0.0
    recall = tp / (tp + fn) if tp + fn else 0.0
    negative_recall = tn / (tn + fp) if tn + fp else 0.0
    f1 = 2 * precision * recall / (precision + recall) if precision + recall else 0.0

    return {
        "model": model_name,
        "accuracy": (tp + tn) / total if total else 0.0,
        "precision": precision,
        "recall": recall,
        "negative_recall": negative_recall,
        "balanced_accuracy": (recall + negative_recall) / 2,
        "f1": f1,
        "roc_auc": roc_auc_evaluator.evaluate(predictions),
        "pr_auc": pr_auc_evaluator.evaluate(predictions),
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn
    }

lr_metrics = evaluate_predictions("Logistic Regression", lr_predictions)
print(lr_metrics)

In [ ]:
# Random Forest
# Create Model
rf = RandomForestClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    numTrees=100
)
# Pipeline
rf_pipeline = Pipeline(stages=[
    payment_indexer,
    assembler,
    scaler,
    rf
])
# Train
rf_model = rf_pipeline.fit(train_df)
# Predict
rf_predictions = rf_model.transform(test_df)
# Evaluate
rf_metrics = evaluate_predictions("Random Forest", rf_predictions)
print(rf_metrics)

In [ ]:
# GBTClassifier
# Create Model
gbt = GBTClassifier(
    featuresCol="scaled_features",
    labelCol="label",
    maxIter=20
)
# Pipeline
gbt_pipeline = Pipeline(stages=[
    payment_indexer,
    assembler,
    scaler,
    gbt
])

# Train
gbt_model = gbt_pipeline.fit(train_df)

# Predict
gbt_predictions = gbt_model.transform(test_df)

# Evaluate
gbt_metrics = evaluate_predictions("GBTClassifier", gbt_predictions)
print(gbt_metrics)

In [ ]:
# Compare Models
model_metrics = [lr_metrics, rf_metrics, gbt_metrics]
metrics_df = spark.createDataFrame(model_metrics)

display(
    metrics_df
    .select(
        "model",
        "accuracy",
        "precision",
        "recall",
        "negative_recall",
        "balanced_accuracy",
        "f1",
        "roc_auc",
        "pr_auc",
        "tp",
        "tn",
        "fp",
        "fn"
    )
    .orderBy(col("balanced_accuracy").desc())
)

trained_models = {
    "Logistic Regression": lr_model,
    "Random Forest": rf_model,
    "GBTClassifier": gbt_model
}

best_metrics = max(
    model_metrics,
    key=lambda metrics: (metrics["balanced_accuracy"], metrics["roc_auc"])
)
best_model_name = best_metrics["model"]
best_model = trained_models[best_model_name]

print("Best model:", best_model_name)
print("Best metrics:", best_metrics)

In [ ]:
# Save Best Model
best_model.write() \
    .overwrite() \
    .save(
        "/Volumes/workspace/default/kthdv&dtdm/models/best_model"
    )

print(f"{best_model_name} saved successfully.")

In [ ]:
# Kiểm tra model đã lưu
display(
    dbutils.fs.ls(
        "/Volumes/workspace/default/kthdv&dtdm/models"
    )
)

In [ ]:
# Load Model Test
from pyspark.ml.pipeline import PipelineModel

loaded_model = PipelineModel.load(
    "/Volumes/workspace/default/kthdv&dtdm/models/best_model"
)

print("Model loaded successfully.")
# Test prediction bằng loaded model
sample_predictions = loaded_model.transform(test_df)

display(
    sample_predictions.select(
        "label",
        "prediction",
        "probability"
    ).limit(10)
)

In [ ]:
# Check feature importance when the selected model supports it.
best_stage = loaded_model.stages[-1]

if hasattr(best_stage, "featureImportances"):
    for feature, importance in zip(
        feature_columns,
        best_stage.featureImportances
    ):
        print(f"{feature}: {importance}")
else:
    print(f"{best_model_name} does not expose featureImportances.")

# Nhận xét Feature Importance của mô hình Random Forest

Sau khi huấn luyện mô hình Random Forest để dự đoán mức độ hài lòng của khách hàng dựa trên review score, nhóm đã tiến hành phân tích độ quan trọng của các đặc trưng (Feature Importance) nhằm xác định những yếu tố ảnh hưởng lớn nhất đến kết quả dự đoán.

Kết quả cho thấy đặc trưng `avg_review_score_customer` có mức độ ảnh hưởng lớn nhất với giá trị khoảng 85.06%. Điều này cho thấy lịch sử đánh giá của khách hàng là yếu tố quan trọng nhất trong việc dự đoán khả năng khách hàng sẽ để lại đánh giá tích cực hay tiêu cực trong các đơn hàng tiếp theo. Nói cách khác, hành vi đánh giá trong quá khứ có tính ổn định và phản ánh rõ xu hướng hài lòng của khách hàng.

Hai đặc trưng có mức ảnh hưởng tiếp theo là `delivery_delay` và `delivery_time`. Trong đó:

* `delivery_delay` phản ánh số ngày giao hàng trễ so với thời gian dự kiến.
* `delivery_time` phản ánh tổng thời gian giao hàng thực tế.

Kết quả cho thấy thời gian giao hàng là yếu tố ảnh hưởng mạnh đến trải nghiệm người dùng. Khi đơn hàng bị giao trễ hoặc thời gian vận chuyển kéo dài, khả năng khách hàng đưa ra đánh giá tiêu cực tăng lên đáng kể. Đây là insight có giá trị thực tiễn cao đối với các hệ thống E-commerce vì nó cho thấy logistics và vận chuyển đóng vai trò quan trọng trong mức độ hài lòng của khách hàng.

Ngoài ra, các đặc trưng như:

* số lượng sản phẩm trong đơn hàng (`number_of_items`)
* thời gian xử lý đơn hàng (`shipping_duration`)
* tổng giá trị đơn hàng (`order_total_value`)
* tổng chi tiêu khách hàng (`customer_total_spent`)

có ảnh hưởng thấp hơn nhưng vẫn góp phần cải thiện khả năng dự đoán của mô hình.

Ngược lại, các đặc trưng liên quan đến thanh toán như:

* loại thanh toán (`payment_type`)
* số kỳ thanh toán (`payment_installments`)

gần như không ảnh hưởng đáng kể đến kết quả dự đoán. Điều này cho thấy khách hàng quan tâm nhiều hơn đến trải nghiệm mua sắm và giao hàng thay vì phương thức thanh toán.

Tổng thể, kết quả Feature Importance cho thấy trải nghiệm hậu mua hàng, đặc biệt là tốc độ và chất lượng giao hàng, là yếu tố quyết định mạnh nhất đến mức độ hài lòng của khách hàng trong hệ thống E-commerce.
